# 09 — Subgraphs

**Learning objective:** decompose a large workflow by responsibility while keeping intentional input/output contracts between parent and child graphs.

Subgraphs improve reuse, local testing, state isolation, and readability when a responsibility has meaningful internal control flow. They are not useful ceremony for every pair of nodes.

## Mental model and topology

```mermaid
flowchart LR
    accTitle: Parent graph with isolated subgraphs
    accDescr: A parent passes documented fields through research, analysis, and review boundaries while each child keeps private working state.

    start_node([Start]) --> research_subgraph[[Research subgraph]]
    research_subgraph --> analysis_subgraph[[Analysis subgraph]]
    analysis_subgraph --> review_subgraph[[Review subgraph]]
    review_subgraph --> end_node([End])
```

Inside research: `retrieve → validate evidence → aggregate`. Inside analysis: `analyze → evaluate → improve` with a bounded loop.

## State and interface contracts

The parent sends only `question` to research and receives `evidence` plus status. Analysis receives evidence and exports a draft plus status. Review receives the draft and exports the final answer plus status. Child-only fields such as raw retrieval results, validation notes, evaluator scratchpad, and reviewer notes never enter parent state. Boundary nodes own these mappings.

In [1]:
from graph_engineering.subgraphs import (
    build_research_subgraph,
    build_subgraph_system,
)

research_graph = build_research_subgraph()
parent_graph = build_subgraph_system()

In [2]:
child_result = research_graph.invoke({"question": "Will heat change demand?"})
print("Child keys:", sorted(child_result))
print("Child status:", child_result["status"])

Child keys: ['evidence', 'question', 'raw_results', 'status', 'validated_results', 'validation_notes']
Child status: validated


In [3]:
parent_result = parent_graph.invoke(
    {"question": "Will heat change demand?", "trace": []}
)
print("Parent keys:", sorted(parent_result))
print("Trace:", parent_result["trace"])
print("Final answer:", parent_result["final_answer"])

Parent keys: ['analysis_status', 'draft', 'evidence', 'final_answer', 'question', 'research_status', 'review_status', 'termination_reason', 'trace']
Trace: ['research_subgraph', 'analysis_subgraph', 'review_subgraph']
Final answer: Reviewed: Analysis of 2 sources with independent-evidence comparison


## Failure considerations

A giant shared state couples children and can leak private fields, create reducer conflicts, and make reuse brittle. Boundary mappings also need explicit failure and cancellation outputs. Do not introduce subgraphs for a tiny workflow, when there is no independent responsibility, or when the abstraction is harder to understand than the nodes it hides.

## What to modify

Add a private research diagnostic and prove it remains absent from parent output. Then change one child implementation without changing its contract.

**Next:** [10 — Multi-Agent Graph](10_multi_agent_graph.ipynb) asks when a computation boundary should be an actual specialist agent.